# 🛡️ Sentinel — FAISS Search Server

Runs the full 40GB IVFFLAT index and exposes it via a Cloudflare tunnel.

**Requirements:**
- Google Colab **Pro** (52GB RAM) or **Pro+** (83GB RAM)
- Runtime type: **High-RAM CPU** (Runtime → Change runtime type → RAM: High)
- Copy your tunnel URL into your FastAPI `.env` as `MODAL_SEARCH_URL`

**After a Colab restart:** just re-run all cells — the index is cached in `/content/artifacts/` so it won't re-download.

## Cell 1 — Install dependencies

In [ ]:
%%capture
!pip install faiss-cpu sentence-transformers fastapi uvicorn huggingface_hub pyngrok nest-asyncio

# Download cloudflared binary
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

print('✅ Dependencies installed')

## Cell 2 — Download FAISS index from HuggingFace (skips if already cached)

In [ ]:
import os
from pathlib import Path
from huggingface_hub import hf_hub_download

ARTIFACTS_DIR = Path('/content/artifacts')
ARTIFACTS_DIR.mkdir(exist_ok=True)

HF_REPO_ID = 'EXANU/antiplagiator-artifacts'
HF_TOKEN   = ''  # ← paste your HF token here if the repo is private

FILES_TO_DOWNLOAD = [
    'faiss_document_index.bin',
    'faiss_metadata.pkl',
]

for filename in FILES_TO_DOWNLOAD:
    dest = ARTIFACTS_DIR / filename
    if dest.exists():
        size_gb = dest.stat().st_size / 1e9
        print(f'⚡ {filename} already cached ({size_gb:.1f} GB) — skipping download')
        continue
    print(f'⬇️  Downloading {filename} ...')
    hf_hub_download(
        repo_id=HF_REPO_ID,
        filename=filename,
        repo_type='dataset',
        token=HF_TOKEN or None,
        local_dir=str(ARTIFACTS_DIR),
    )
    print(f'✅ {filename} downloaded')

print('\n📦 All artifacts ready.')

## Cell 3 — Load index + embedding model into RAM

In [ ]:
import faiss
import pickle
import time
from sentence_transformers import SentenceTransformer

NPROBE = 20
MODEL_NAME = 'BAAI/bge-base-en-v1.5'

print('📂 Loading FAISS index into RAM (this takes 1–3 minutes for 40GB)...')
t0 = time.monotonic()
index = faiss.read_index(str(ARTIFACTS_DIR / 'faiss_document_index.bin'))
index.nprobe = NPROBE
print(f'✅ Index loaded in {time.monotonic()-t0:.1f}s — {index.ntotal:,} vectors')

print('📂 Loading metadata...')
with open(ARTIFACTS_DIR / 'faiss_metadata.pkl', 'rb') as f:
    metadata = pickle.load(f)
print(f'✅ Metadata loaded — {len(metadata):,} rows')

print(f'\n🤖 Loading embedding model: {MODEL_NAME} ...')
model = SentenceTransformer(MODEL_NAME, device='cpu')
print('✅ Model ready')

# Quick RAM check
import psutil
ram = psutil.virtual_memory()
print(f'\n💾 RAM: {ram.used/1e9:.1f}GB used / {ram.total/1e9:.1f}GB total ({ram.percent}%)')

## Cell 4 — Start FastAPI server + Cloudflare tunnel

In [ ]:
import asyncio
import subprocess
import threading
import time
import re
import numpy as np
import nest_asyncio
import uvicorn
from fastapi import FastAPI
from fastapi.responses import JSONResponse
from typing import Any

nest_asyncio.apply()

# ── FastAPI app ──────────────────────────────────────────────────────────────

api = FastAPI(title='Sentinel FAISS Service')

@api.get('/health')
def health():
    return {
        'status': 'ok',
        'total_vectors': index.ntotal,
        'metadata_rows': len(metadata),
        'nprobe': index.nprobe,
    }

@api.post('/search')
def search(request: dict) -> dict:
    """
    Body: { chunks, top_k, threshold, self_arxiv_id }
    Same interface as the Modal microservice.
    """
    chunks        = request.get('chunks', [])
    top_k         = int(request.get('top_k', 5))
    threshold     = float(request.get('threshold', 0.85))
    self_arxiv_id = request.get('self_arxiv_id')

    if not chunks:
        return {'error': 'No chunks provided', 'results': []}

    t0 = time.monotonic()

    vectors = model.encode(
        chunks,
        convert_to_numpy=True,
        normalize_embeddings=True,
        batch_size=32,
        show_progress_bar=False,
    ).astype('float32')

    similarities, db_indices = index.search(vectors, k=top_k)

    results: list[dict[str, Any]] = []
    for chunk_idx, (scores, indices) in enumerate(zip(similarities, db_indices)):
        hits = []
        for sim, db_idx in zip(scores.tolist(), indices.tolist()):
            if db_idx < 0 or sim < threshold or db_idx >= len(metadata):
                continue
            meta = metadata[db_idx]
            if self_arxiv_id and meta.get('arxiv_id') == self_arxiv_id:
                continue
            hits.append({
                'db_idx':       db_idx,
                'similarity':   round(float(sim), 4),
                'arxiv_id':     meta.get('arxiv_id', 'N/A'),
                'chunk_id':     meta.get('chunk_id', -1),
                'title':        meta.get('title', 'N/A'),
                'source_type':  meta.get('source_type', 'unknown'),
                'top_category': meta.get('top_category', ''),
            })
        results.append({'chunk_idx': chunk_idx, 'hits': hits})

    return {'results': results, 'timing_s': round(time.monotonic() - t0, 3)}


# ── Start uvicorn in a background thread ────────────────────────────────────

PORT = 8000

def run_server():
    uvicorn.run(api, host='0.0.0.0', port=PORT, log_level='warning')

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(2)  # let uvicorn start
print(f'✅ FastAPI running on http://localhost:{PORT}')


# ── Start Cloudflare tunnel ──────────────────────────────────────────────────

tunnel_proc = subprocess.Popen(
    ['/usr/local/bin/cloudflared', 'tunnel', '--url', f'http://localhost:{PORT}'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

print('⏳ Waiting for Cloudflare tunnel URL...')
tunnel_url = None
for line in tunnel_proc.stdout:
    match = re.search(r'https://[a-z0-9\-]+\.trycloudflare\.com', line)
    if match:
        tunnel_url = match.group(0)
        break

print()
print('=' * 60)
print(f'🌐 PUBLIC URL: {tunnel_url}')
print(f'🔍 Health:     {tunnel_url}/health')
print(f'🔎 Search:     POST {tunnel_url}/search')
print('=' * 60)
print()
print('👉 Copy the URL above into your FastAPI .env:')
print(f'   MODAL_SEARCH_URL={tunnel_url}/search')
print()
print('⚠️  This URL changes every time you restart. Keep this cell running.')

## Cell 5 — (Optional) Quick local test
Run this after Cell 4 to confirm the server responds correctly.

In [ ]:
import requests, json

# Test health
r = requests.get(f'http://localhost:{PORT}/health')
print('Health:', json.dumps(r.json(), indent=2))

# Test search with a dummy chunk
r = requests.post(f'http://localhost:{PORT}/search', json={
    'chunks': ['deep learning neural network gradient descent optimization'],
    'top_k': 3,
    'threshold': 0.5,
})
data = r.json()
print(f'\nSearch returned {len(data["results"][0]["hits"])} hits in {data["timing_s"]}s')
for hit in data['results'][0]['hits']:
    print(f'  [{hit["similarity"]:.3f}] {hit["arxiv_id"]} — {hit["title"][:60]}')